# 多模态有害模因检测 · 实验结果可视化（Qwen3.8-27B）

输入：`results/*_full.jsonl`（每条样本一行）和 `*_errors.jsonl`（预测错误的样本）。
输出：论文可用的图（PNG 300 dpi + PDF 矢量）与表（CSV / LaTeX），保存在 `figures/`。

**使用方法**：改下面「0. 配置」里的路径，然后 *Run All*。之后再测别的模型，只需把 `MODEL_FILE` 换掉即可。

> ⚠️ `gt_type` 的数字含义（1–4）是我从数据里**推断**的（预测出的类型名与 gt_type 的对应关系），请对照你的标注手册确认；不对就改 `TYPE_NAMES`。

## 0. 配置

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sci_viz as pl
from sci_viz import stats
from sci_viz.utils import pubstyle as ps

DATA_DIR   = Path("results")                       # 结果文件夹
MODEL_FILE = "Qwen3.8-27B_full.jsonl"              # 完整结果
ERR_FILE   = "Qwen3.8-27B_errors.jsonl"            # 错误样本（可选，没有就设 None）
FIG_DIR    = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

# gt_type → 类型名（⚠ 推断的，请核对）。0 = 无害
TYPE_NAMES = {0: "harmless", 1: "targeted_harm", 2: "sexual_innuendo", 3: "general_offense", 4: "dispirited_culture"}
# 图上显示的名字（想用中文就改这里；宋体已由 pl 全局设置好）
SHOW = {"harmless": "Harmless", "targeted_harm": "Targeted harm", "sexual_innuendo": "Sexual innuendo",
        "general_offense": "General offense", "dispirited_culture": "Dispirited culture"}

pl.set_theme("ocean")                              # 全篇论文统一主题色；换主题只改这一行
DPI = 300

## 1. 读取与数据检查

In [ ]:
df = pl.read_jsonl(DATA_DIR / MODEL_FILE)
df["gt_name"]   = df["gt_type"].map(TYPE_NAMES)
df["pred_name"] = df["pred_type"].fillna("harmless")             # 判无害时 pred_type 为空
df["y_true"]    = df["gt_label"].astype(int)                      # 1 = 有害
df["y_pred"]    = (df["pred_label"] == "harmful").astype(int)

print(f"样本数 {len(df)}，唯一 sample_id {df.sample_id.nunique()}，模型 {df.model.unique()}")
print("真实有害占比 %.1f%%" % (100 * df.y_true.mean()))
print("pred_label 与 correct 是否自洽:", bool(((df.y_true == df.y_pred) == df.correct).all()))
print("API/解析错误行数:", int(df["error"].notna().sum()))
df.gt_name.value_counts().rename("n").to_frame().T

## 2. 总体指标（带 95% 置信区间）
准确率 / 精确率 / 召回率 / 特异度用 Wilson 区间，F1 / 平衡准确率 / MCC 用样本级 bootstrap（2000 次，固定种子可复现）。
**务必同时报告多数类基线**：样本不平衡时，准确率本身没有说服力。

In [ ]:
m = stats.binary_metrics(df.y_true, df.y_pred, positive=1)
base = stats.majority_baseline(df.y_true)
cnt = m.attrs["counts"]
print("TP/FP/FN/TN =", cnt, "| 多数类基线准确率 = %.1f%%" % (100 * base))
tab = pd.DataFrame({"Metric": m.index,
                    "Value (%) [95% CI]": [stats.format_ci(r.value, r.lo, r.hi) if i != "MCC"
                                           else stats.format_ci(r.value, r.lo, r.hi, digits=2, percent=False)
                                           for i, r in m.iterrows()]})
tab

## 3. 图 1：二分类混淆矩阵 + 各指标

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=pl.figsize("double", 0.42), gridspec_kw=dict(width_ratios=[1, 1.25]))
cm2 = stats.confusion_table(df.y_true, df.y_pred, labels=[0, 1], pred_labels=[0, 1])
pl.confusion_heatmap(cm2, a, labels=["Harmless", "Harmful"], pred_labels=["Harmless", "Harmful"], title="Binary confusion matrix")
prop = {k: (int(m.loc[k, "k"]), int(m.loc[k, "n"])) for k in ["Accuracy", "Precision", "Recall", "Specificity"]}
pl.rate_bar(prop, b, baseline=base, baseline_label="Majority baseline", show_n=False, title="Overall metrics", rate_label="Score (%)")
for ax_, t in zip((a, b), "ab"): ps.panel_label(ax_, t)
fig.tight_layout(); pl.save(fig, FIG_DIR / "fig1_overall"); fig

## 4. 图 2：按真实类别的正确率
有害各子类 = 召回率（被识别为“有害”的比例）；Harmless = 特异度（被识别为“无害”的比例）。

In [ ]:
order = [TYPE_NAMES[i] for i in range(5)]
fig, ax = plt.subplots(figsize=pl.figsize("1.5", 0.62))
pl.rate_bar(df, ax, by="gt_name", value="correct", order=order, labels=[SHOW[o] for o in order],
            baseline=base, baseline_label="Majority baseline", title="Detection rate by ground-truth type",
            rate_label="Correct (%)")
fig.tight_layout(); pl.save(fig, FIG_DIR / "fig2_by_type"); fig

In [ ]:
per_type = stats.rate_table(df, by="gt_name", value="correct", order=order)
per_type["Correct (%) [95% CI]"] = [stats.format_ci(r.rate, r.lo, r.hi) for r in per_type.itertuples()]
per_type[["group", "k", "n", "Correct (%) [95% CI]"]].rename(columns={"group": "gt_type"})

## 5. 图 3：类型级混淆矩阵（真实 5 类 × 预测 5 类）
行归一化：每行 = 该真实类别被模型分到各类型的比例。对角线 = 类型也判对。

In [ ]:
pred_order = [TYPE_NAMES[i] for i in range(5)]
cm5 = stats.confusion_table(df.gt_name, df.pred_name, labels=order, pred_labels=pred_order)
fig, ax = plt.subplots(figsize=pl.figsize("1.5", 0.85))
pl.confusion_heatmap(cm5, ax, labels=[SHOW[o] for o in order], pred_labels=[SHOW[o] for o in pred_order],
                     title="Type-level confusion matrix", xtick_rotation=30, fontsize=6.5)
fig.tight_layout(); pl.save(fig, FIG_DIR / "fig3_type_confusion"); fig

## 6. 图 4：错误结构（模型“错”在哪）
左：误报（无害被判有害）被分到了哪些有害类型；右：漏报（有害被判无害）来自哪些真实类型。

In [ ]:
fp = df[(df.y_true == 0) & (df.y_pred == 1)].pred_name.value_counts().reindex(order[1:], fill_value=0)
fn = df[(df.y_true == 1) & (df.y_pred == 0)].gt_name.value_counts().reindex(order[1:], fill_value=0)
cols = pl.palettes.get_palette(None, "rate_bar", n=4)
fig, axes = plt.subplots(1, 2, figsize=pl.figsize("double", 0.38))
for ax, s, t, xl in ((axes[0], fp, f"False positives (n={fp.sum()})", "Predicted type"),
                     (axes[1], fn, f"False negatives (n={fn.sum()})", "Ground-truth type")):
    ax.barh(range(4), s.values, color=cols, alpha=.5, edgecolor=cols, linewidth=2)
    for i, v in enumerate(s.values): ax.text(v + s.max() * .02, i, f"{v} ({100 * v / s.sum():.1f}%)", va="center", fontsize=7)
    ax.set_yticks(range(4)); ax.set_yticklabels([SHOW[o] for o in order[1:]]); ax.invert_yaxis()
    ax.set_xlim(0, s.max() * 1.35); ax.set_xlabel("Count"); ax.set_title(t); ps.clean_axis(ax)
for ax_, t in zip(axes, "ab"): ps.panel_label(ax_, t)
fig.tight_layout(); pl.save(fig, FIG_DIR / "fig4_error_structure"); fig

## 7. 组合图（论文里的 Figure：双栏宽，三联）
a 二分类混淆矩阵 · b 各类型正确率 · c 类型级混淆矩阵 · d 误报流向

In [ ]:
fig = plt.figure(figsize=pl.figsize("double", 0.80))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.2], hspace=0.55, wspace=0.8)
a, b, c, d = (fig.add_subplot(g) for g in gs)
pl.confusion_heatmap(cm2, a, labels=["Harmless", "Harmful"], pred_labels=["Harmless", "Harmful"], cbar=False, title="Binary confusion matrix")
pl.rate_bar(df, b, by="gt_name", value="correct", order=order, labels=[SHOW[o] for o in order], show_n=False,
            baseline=base, baseline_label="Majority baseline", rate_label="Correct (%)", title="Correct rate by type")
pl.confusion_heatmap(cm5, c, labels=[SHOW[o] for o in order], pred_labels=[SHOW[o] for o in pred_order],
                     xtick_rotation=30, fontsize=6, cbar=False, title="Type-level confusion matrix")
d.barh(range(4), fp.values, color=cols, alpha=.5, edgecolor=cols, linewidth=2)
for i, v in enumerate(fp.values): d.text(v + fp.max() * .02, i, f"{v}", va="center", fontsize=7)
d.set_yticks(range(4)); d.set_yticklabels([SHOW[o] for o in order[1:]]); d.invert_yaxis()
d.set_xlim(0, fp.max() * 1.2); d.set_xlabel("Count"); d.set_title("False positives by predicted type"); ps.clean_axis(d)
for ax_, t in zip((a, b, c, d), "abcd"): ps.panel_label(ax_, t)
pl.save(fig, FIG_DIR / "fig_composite", bbox_inches="tight"); fig

## 8. 论文表格（CSV + LaTeX）

In [ ]:
tab.to_csv(FIG_DIR / "table_overall.csv", index=False)
per = per_type[["group", "n", "Correct (%) [95% CI]"]].rename(columns={"group": "Type", "n": "N"})
per.to_csv(FIG_DIR / "table_by_type.csv", index=False)
latex = tab.to_latex(index=False, escape=True, column_format="lc", caption="Overall performance (95\\% CI).", label="tab:overall")
(FIG_DIR / "table_overall.tex").write_text(latex, encoding="utf-8")
print(latex)

## 9. 典型错误案例（写 Error analysis 用）
从错误文件里按“错误方向 × 类型”各抽几条，`reason` 是模型给的理由。

In [ ]:
err = pl.read_jsonl(DATA_DIR / ERR_FILE) if ERR_FILE else df[~df.correct]
err["gt_name"] = err["gt_type"].map(TYPE_NAMES); err["pred_name"] = err["pred_type"].fillna("harmless")
err["direction"] = np.where(err["gt_label"].astype(int) == 0, "FP (harmless→harmful)", "FN (harmful→harmless)")
print("错误样本数 %d（与 full 中 correct=False 一致: %s）" % (len(err), len(err) == (~df.correct).sum()))
cases = (err.groupby(["direction", "gt_name"], group_keys=False).head(2)
            .sort_values(["direction", "gt_name"])[["sample_id", "direction", "gt_name", "pred_name", "reason"]])
cases.to_csv(FIG_DIR / "error_cases.csv", index=False)
pd.set_option("display.max_colwidth", 80); cases

## 10. 要点（自动生成，随数据更新）

In [ ]:
r = m.loc
print(f"· 准确率 {100*r['Accuracy','value']:.1f}%，低于多数类基线 {100*base:.1f}%；MCC = {r['MCC','value']:.2f}。")
print(f"· 召回 {100*r['Recall','value']:.1f}% 高、精确率 {100*r['Precision','value']:.1f}% 低 → 模型倾向于过度判有害（FP={cnt['FP']}，FN={cnt['FN']}）。")
print(f"· 无害样本的特异度仅 {100*r['Specificity','value']:.1f}%。")
print(f"· 误报最多被归为 {fp.idxmax()}（{fp.max()} 条）。")